In [ ]:
!pip -q install amplpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.4 MB/s eta 0:00:00


In [ ]:
# 1. Instalación e inicialización de AMPL con tu licencia Community Edition
from amplpy import ampl_notebook

# Inicializa AMPL en Colab con tu UUID (Community Edition) Inicializar AMPL
ampl = ampl_notebook(
    modules=["highs", "cbc", "gurobi", "cplex"],  # solvers disponibles
    license_uuid="936b618d-a013-406f-9809-49679f557c26"
)

Licensed to AMPL Academic Community Edition License for <m.godoyseplveda@uandresbello.edu>.


In [ ]:
%%writefile 13_1_2.mod
set V;                               # vertices 1..n
param d {V,V} >= 0;                  # distancia simetrica

var x {i in V, j in V : i<j} binary; # 1 si arista (i,j) esta en el tour
var u {V} >= 0, <= card(V)-1;        # variables MTZ (evitan subtours)

minimize Z : sum {i in V, j in V : i<j} d[i,j] * x[i,j];

# grado 2
s.t. Deg {i in V} : sum {j in V : j<i} x[j,i] + sum {j in V : j>i} x[i,j] = 2;

# cortes MTZ  (solo para i,j != 1)
s.t. MTZ {i in V diff {1}, j in V diff {1} : i!=j and i<j} :
          u[i] - u[j] + card(V)*x[i,j] <= card(V)-1;


Overwriting 13_1_2.mod


In [ ]:
%%writefile 13_1_2.dat
set V := 1 2 3 4 5 6 7 ;

param d :  1   2   3   4   5   6   7 :=
     1    0  10  15  20  10  25  30
     2   10   0  35  25  17  30  28
     3   15  35   0  30   5  12  20
     4   20  25  30   0  20  10  20
     5   10  17   5  20   0  20  12
     6   25  30  12  10  20   0  10
     7   30  28  20  20  12  10   0 ;


Overwriting 13_1_2.dat


In [ ]:
# Reseteamos el entorno AMPL
ampl.reset()

# Cargamos el modelo CORREGIDO y los datos
ampl.read('13_1_2.mod')
ampl.readData('13_1_2.dat')

# Configuramos el solver y resolvemos
ampl.option['solver'] = 'highs'
ampl.solve()

# Mostramos resultados
ampl.display('Z')
print("\nAristas en el tour óptimo:")
for i in ampl.getSet('V'):
    for j in ampl.getSet('V'):
        if i < j and ampl.getVariable('x')[i,j].value() > 0.5:
            print(f"{i}-{j} (distancia: {ampl.getParameter('d')[i,j]})")

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 87
15 simplex iterations
1 branching nodes
Z = 87


Aristas en el tour óptimo:
1-2 (distancia: 10)
1-3 (distancia: 15)
2-4 (distancia: 25)
3-5 (distancia: 5)
4-6 (distancia: 10)
5-7 (distancia: 12)
6-7 (distancia: 10)


In [ ]:
# Reconstrucción del tour completo
print("\nReconstruyendo el tour completo...")
def reconstruir_tour(ampl_instance):
    nodos = list(ampl_instance.getSet('V'))
    conexiones = {i: [] for i in nodos}

    for i in nodos:
        for j in nodos:
            if i < j and ampl_instance.getVariable('x')[i,j].value() > 0.5:
                conexiones[i].append(j)
                conexiones[j].append(i)

    tour = [1]  # Empezamos desde la ciudad 1
    visitados = set(tour)

    while len(tour) < len(nodos):
        ultimo = tour[-1]
        for vecino in conexiones[ultimo]:
            if vecino not in visitados:
                tour.append(vecino)
                visitados.add(vecino)
                break

    tour.append(1)  # Cerramos el ciclo
    return tour


Reconstruyendo el tour completo...


In [ ]:
tour_optimo = reconstruir_tour(ampl)
print("\nTour óptimo encontrado:")
print(" -> ".join(map(str, tour_optimo)))

# Verificación de distancia
distancia = sum(ampl.getParameter('d')[tour_optimo[i-1], tour_optimo[i]]
             for i in range(1, len(tour_optimo)))
print(f"Distancia total confirmada: {distancia}")


Tour óptimo encontrado:
1 -> 2 -> 4 -> 6 -> 7 -> 5 -> 3 -> 1
Distancia total confirmada: 87


In [ ]:
def calcular_distancia(tour, distancias):
    return sum(distancias[tour[i]-1][tour[i+1]-1] for i in range(len(tour)-1)) + distancias[tour[-1]-1][tour[0]-1]

def two_opt(tour, distancias):
    mejor_tour = tour.copy()
    mejor_distancia = calcular_distancia(tour, distancias)
    mejorado = True

    while mejorado:
        mejorado = False
        for i in range(1, len(tour)-2):
            for j in range(i+1, len(tour)-1):
                nuevo_tour = tour[:i] + tour[i:j+1][::-1] + tour[j+1:]
                nueva_distancia = calcular_distancia(nuevo_tour, distancias)
                if nueva_distancia < mejor_distancia:
                    mejor_tour = nuevo_tour
                    mejor_distancia = nueva_distancia
                    mejorado = True
                    break
            if mejorado:
                break
    return mejor_tour, mejor_distancia

# Datos de distancia
distancias = [
    [0, 10, 15, 20, 10, 25, 30],
    [10, 0, 35, 25, 17, 30, 28],
    [15, 35, 0, 30, 5, 12, 20],
    [20, 25, 30, 0, 20, 10, 20],
    [10, 17, 5, 20, 0, 20, 12],
    [25, 30, 12, 10, 20, 0, 10],
    [30, 28, 20, 20, 12, 10, 0]
]

# Tour inicial para parte b)
tour_inicial = [1, 2, 4, 5, 6, 7, 3, 1]
tour_2opt, distancia_2opt = two_opt(tour_inicial, distancias)

print("\nResultado algoritmo 2-opt (parte b):")
print(f"Tour inicial: {' -> '.join(map(str, tour_inicial))} (distancia: {calcular_distancia(tour_inicial, distancias)})")
print(f"Tour mejorado: {' -> '.join(map(str, tour_2opt))} (distancia: {distancia_2opt})")


Resultado algoritmo 2-opt (parte b):
Tour inicial: 1 -> 2 -> 4 -> 5 -> 6 -> 7 -> 3 -> 1 (distancia: 120)
Tour mejorado: 1 -> 2 -> 5 -> 4 -> 6 -> 7 -> 3 -> 1 (distancia: 102)
